<a href="https://colab.research.google.com/github/AyaAbdElNaem/AI_Tools/blob/main/Final_XLSTM_Gannet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Cell 1: Libraries, Device Configuration, and Dynamic Model Architecture

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0.0)

class NativesLSTMLayer(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(NativesLSTMLayer, self).__init__()
        self.hidden_dim = hidden_dim
        self.W_x = nn.Linear(input_dim, 4 * hidden_dim, bias=True)
        self.W_h = nn.Linear(hidden_dim, 4 * hidden_dim, bias=False)
        init_weights(self.W_x)
        init_weights(self.W_h)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        h = torch.zeros(batch_size, self.hidden_dim, device=x.device)
        c = torch.zeros(batch_size, self.hidden_dim, device=x.device)
        outputs = []
        for t in range(seq_len):
            x_t = x[:, t, :]
            gates = self.W_x(x_t) + self.W_h(h)
            i_gate, f_gate, c_gate, o_gate = gates.chunk(4, dim=1)

            i_t = torch.exp(torch.clamp(i_gate, -5.0, 5.0))
            f_t = torch.exp(torch.clamp(f_gate, -5.0, 5.0))

            c_tilde = torch.tanh(c_gate)
            c = f_t * c + i_t * c_tilde
            o_t = torch.sigmoid(o_gate)
            h = o_t * torch.tanh(c)
            outputs.append(h.unsqueeze(1))
        return torch.cat(outputs, dim=1)

class JointxLSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim, hidden_dim, num_layers=2, dropout_prob=0.2):
        super(JointxLSTMAutoencoder, self).__init__()
        self.num_layers = num_layers

        # Dynamic Encoder Stack
        self.encoder_layers = nn.ModuleList()
        current_dim = input_dim
        next_dim = hidden_dim
        for i in range(num_layers):
            self.encoder_layers.append(NativesLSTMLayer(current_dim, next_dim))
            current_dim = next_dim
            if i == 0 and num_layers > 1:
                next_dim = max(8, hidden_dim // 2)

        self.bottleneck = nn.Linear(current_dim, latent_dim)
        self.encoder_dropout = nn.Dropout(dropout_prob)

        # Dynamic Decoder Stack
        self.decoder_layers = nn.ModuleList()
        current_dim = latent_dim
        next_dim = max(8, hidden_dim // 2) if num_layers > 1 else hidden_dim
        for i in range(num_layers):
            if i == num_layers - 1:
                next_dim = hidden_dim
            self.decoder_layers.append(NativesLSTMLayer(current_dim, next_dim))
            current_dim = next_dim

        self.reconstruct = nn.Linear(hidden_dim, input_dim)
        self.decoder_dropout = nn.Dropout(dropout_prob)
        self.predictor_head = nn.Linear(latent_dim, 1)

        init_weights(self.bottleneck)
        init_weights(self.reconstruct)
        init_weights(self.predictor_head)

    def forward(self, x):
        encoded = x
        for layer in self.encoder_layers:
            encoded = layer(encoded)
            encoded = self.encoder_dropout(encoded)
        latent = self.bottleneck(encoded[:, -1, :])

        decoded_input = latent.unsqueeze(1).repeat(1, x.size(1), 1)
        decoded = decoded_input
        for layer in self.decoder_layers:
            decoded = layer(decoded)
            decoded = self.decoder_dropout(decoded)
        reconstructed_output = torch.sigmoid(self.reconstruct(decoded))
        predicted_carbon = self.predictor_head(latent)

        return reconstructed_output, predicted_carbon, latent

#Cell 2: Data Preprocessing and Splitting

In [2]:
FILE_PATH = '/content/rural_carbon_dataset.csv'
df = pd.read_csv(FILE_PATH)
df_processed = df.copy()

# Feature Engineering
crop_encoder = LabelEncoder()
df_processed['Crop_Type_Encoded'] = crop_encoder.fit_transform(df_processed['Crop_Type'])
df_processed['Month_sin'] = np.sin(2 * np.pi * df_processed['Month'] / 12)
df_processed['Month_cos'] = np.cos(2 * np.pi * df_processed['Month'] / 12)
df_processed['Livestock_Total'] = df_processed['Livestock_Cows'] + df_processed['Livestock_Pigs']
df_processed['Energy_per_Area'] = df_processed['Household_Energy_kWh'] / (df_processed['Crop_Area_ha'] + 1)
df_processed['Fertilizer_per_Area'] = df_processed['Fertilizer_Usage_kg'] / (df_processed['Crop_Area_ha'] + 1)

feature_cols = [
    'Month_sin', 'Month_cos', 'Crop_Type_Encoded', 'Crop_Area_ha', 'Livestock_Total',
    'Household_Energy_kWh', 'Renewable_Energy_Fraction', 'Temperature_C',
    'Rainfall_mm', 'Energy_per_Area', 'Fertilizer_per_Area'
]

X = df_processed[feature_cols].values.astype(np.float32)
y = df_processed['Carbon_Emission_tCO2'].values.astype(np.float32).reshape(-1, 1)

# Split into Train and Validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalization
scaler_X = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)

scaler_y = MinMaxScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_val_scaled = scaler_y.transform(y_val)

X_train_3d = np.expand_dims(X_train_scaled, axis=1)
X_val_3d = np.expand_dims(X_val_scaled, axis=1)

val_inputs_X = torch.tensor(X_val_3d).to(device)
val_targets_y = torch.tensor(y_val_scaled).to(device)
feat_dim = X_train_3d.shape[2]

print("Data preprocessing completed successfully.")

Data preprocessing completed successfully.


#Cell 3: Gannet Optimization Algorithm Phase (Independent Metaheuristic Stage)

In [3]:
# Gannet Optimization Algorithm for Hyperparameter Tuning
# Search Space Dimensions:
# [LR, Hidden_Size, Latent_Dim, Dropout, Weight_Decay, Batch_Size, Num_Layers, Alpha]
lb = np.array([1e-4, 16,  4, 0.0, 1e-6, 16,  1, 0.1]) # Lower bounds
ub = np.array([1e-2, 64, 16, 0.5, 1e-3, 128, 3, 2.0]) # Upper bounds

pop_size = 10
max_iter = 5
feat_dim = X_train_3d.shape[2]

def evaluate_fitness(position):
    # Decode position values
    lr = float(position[0])
    hidden_size = int(np.round(position[1]))
    latent_dim = int(np.round(position[2]))
    dropout = float(position[3])
    weight_decay = float(position[4])
    batch_size = int(np.round(position[5]))
    num_layers = int(np.round(position[6]))
    alpha = float(position[7])

    # Fast evaluation dataloader
    t_dataset = TensorDataset(torch.tensor(X_train_3d), torch.tensor(y_train_scaled))
    t_loader = DataLoader(t_dataset, batch_size=batch_size, shuffle=True)

    # Instantiate dynamic model
    eval_model = JointxLSTMAutoencoder(
        input_dim=feat_dim, latent_dim=latent_dim, hidden_dim=hidden_size,
        num_layers=num_layers, dropout_prob=dropout
    ).to(device)

    criterion_r = nn.MSELoss()
    criterion_p = nn.MSELoss()
    opt = optim.Adam(eval_model.parameters(), lr=lr, weight_decay=weight_decay)

    # Short optimization training loop for fitness assignment
    for epoch in range(3):
        eval_model.train()
        for bx, by in t_loader:
            bx, by = bx.to(device), by.to(device)
            opt.zero_grad()
            r_out, p_out, _ = eval_model(bx)
            loss = criterion_r(r_out, bx) + (alpha * criterion_p(p_out, by))
            loss.backward()
            nn.utils.clip_grad_norm_(eval_model.parameters(), 5.0)
            opt.step()

    # Evaluation on Validation set
    eval_model.eval()
    with torch.no_grad():
        v_r, v_p, _ = eval_model(val_inputs_X)
        val_loss = criterion_r(v_r, val_inputs_X).item() + (alpha * criterion_p(v_p, val_targets_y).item())
    return val_loss

# Initialize Gannet Population
gannet_positions = np.random.uniform(lb, ub, (pop_size, len(lb)))
fitness_scores = np.array([evaluate_fitness(p) for p in gannet_positions])

best_idx = np.argmin(fitness_scores)
best_gannet_score = fitness_scores[best_idx]
best_gannet_position = gannet_positions[best_idx].copy()

print("Executing Gannet Optimization Strategy...")
for iteration in range(max_iter):
    for i in range(pop_size):
        # Gannet Exploration and Exploitation mathematical updating mechanics
        t = 1 - (iteration / max_iter)
        c = 0.2 * (t ** 2)
        v = np.random.randn(*lb.shape)

        if np.random.rand() < 0.5:
            # Dive exploration phase
            gannet_positions[i] = gannet_positions[i] + c * v * (gannet_positions[i] - best_gannet_position)
        else:
            # Trajectory exploitation phase
            gannet_positions[i] = best_gannet_position + c * np.random.rand() * (best_gannet_position - gannet_positions[i])

        # Bound enforcement
        gannet_positions[i] = np.clip(gannet_positions[i], lb, ub)

        # Re-evaluate
        fit = evaluate_fitness(gannet_positions[i])
        if fit < fitness_scores[i]:
            fitness_scores[i] = fit
            if fit < best_gannet_score:
                best_gannet_score = fit
                best_gannet_position = gannet_positions[i].copy()

    print(f"Gannet Iteration [{iteration+1}/{max_iter}] -> Best Discovered Fitness: {best_gannet_score:.6f}")

# Extract Optimized Global Best Parameters
best_lr = float(best_gannet_position[0])
best_hidden_size = int(np.round(best_gannet_position[1]))
best_latent_dim = int(np.round(best_gannet_position[2]))
best_dropout = float(best_gannet_position[3])
best_weight_decay = float(best_gannet_position[4])
best_batch_size = int(np.round(best_gannet_position[5]))
best_num_layers = int(np.round(best_gannet_position[6]))
best_alpha = float(best_gannet_position[7])

print("\nGannet Search Completed. Optimal Hyperparameters Parsed.")

Executing Gannet Optimization Strategy...
Gannet Iteration [1/5] -> Best Discovered Fitness: 0.031703
Gannet Iteration [2/5] -> Best Discovered Fitness: 0.025698
Gannet Iteration [3/5] -> Best Discovered Fitness: 0.025698
Gannet Iteration [4/5] -> Best Discovered Fitness: 0.025698
Gannet Iteration [5/5] -> Best Discovered Fitness: 0.025698

Gannet Search Completed. Optimal Hyperparameters Parsed.


#Cell 4: Structured Adam Training Pipeline

In [8]:
# Construct final optimized train loader
train_dataset = TensorDataset(torch.tensor(X_train_3d), torch.tensor(y_train_scaled))
train_loader = DataLoader(train_dataset, batch_size=best_batch_size, shuffle=True)

# Build optimal architectural network
model = JointxLSTMAutoencoder(
    input_dim=feat_dim, latent_dim=best_latent_dim, hidden_dim=best_hidden_size,
    num_layers=best_num_layers, dropout_prob=best_dropout
).to(device)

criterion_recon = nn.MSELoss()
criterion_pred = nn.MSELoss()

# Primary Adam Optimizer paired with Weight Decay from Gannet Phase
optimizer = optim.Adam(model.parameters(), lr=best_lr, weight_decay=best_weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

epochs = 100
best_val_loss = float('inf')
patience, patience_counter = 20, 0
checkpoint_path = 'best_joint_xlstm_ae_checkpoint.pth'

print(f"Beginning Deep Architecture Training Sequence using Optimal Gannet Configurations...")
print("-" * 90)

for epoch in range(epochs):
    # Training Loop
    model.train()
    train_total_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        recon_out, pred_carbon, _ = model(batch_x)

        loss_recon = criterion_recon(recon_out, batch_x)
        loss_pred = criterion_pred(pred_carbon, batch_y)
        loss_total = loss_recon + (best_alpha * loss_pred)

        loss_total.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_total_loss += loss_total.item() * batch_x.size(0)
    train_total_loss /= len(train_loader.dataset)

    # Validation Loop
    model.eval()
    with torch.no_grad():
        val_recon, val_pred, _ = model(val_inputs_X)
        val_recon_loss = criterion_recon(val_recon, val_inputs_X).item()
        val_pred_loss = criterion_pred(val_pred, val_targets_y).item()
        val_total_loss = val_recon_loss + (best_alpha * loss_pred.item()) # Safe scalar reference

    scheduler.step(val_total_loss)

    # Checkpoint and Early Stopping Check
    if val_total_loss < best_val_loss:
        best_val_loss = val_total_loss
        patience_counter = 0

        # FIX: Convert numpy config to PyTorch Tensor before saving to eliminate UnpicklingError completely
        safe_config_tensor = torch.tensor(best_gannet_position, dtype=torch.float32)

        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'best_val_loss': float(best_val_loss),
            'config_tensor': safe_config_tensor
        }, checkpoint_path)
    else:
        patience_counter += 1

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:03d}/{epochs}] -> Train Loss: {train_total_loss:.6f} | Val Recon Loss: {val_recon_loss:.6f} | Val Pred Loss: {val_pred_loss:.6f} | Total Val Loss: {val_total_loss:.6f}")

    if patience_counter >= patience:
        print(f"Early stopping triggered at epoch {epoch+1} to safeguard generalization.")
        break

print("-" * 90)
print("Optimized Model Training Phase Finished.")

Beginning Deep Architecture Training Sequence using Optimal Gannet Configurations...
------------------------------------------------------------------------------------------
Epoch [001/100] -> Train Loss: 0.109013 | Val Recon Loss: 0.022944 | Val Pred Loss: 0.011484 | Total Val Loss: 0.040149
Epoch [010/100] -> Train Loss: 0.033892 | Val Recon Loss: 0.002094 | Val Pred Loss: 0.012418 | Total Val Loss: 0.020083
Epoch [020/100] -> Train Loss: 0.032720 | Val Recon Loss: 0.003065 | Val Pred Loss: 0.015414 | Total Val Loss: 0.046524
Epoch [030/100] -> Train Loss: 0.029847 | Val Recon Loss: 0.001386 | Val Pred Loss: 0.011493 | Total Val Loss: 0.037204
Early stopping triggered at epoch 34 to safeguard generalization.
------------------------------------------------------------------------------------------
Optimized Model Training Phase Finished.


#Cell 5: Full Performance Evaluation Metrics

In [9]:
def calculate_mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

if os.path.exists(checkpoint_path):
    # Pure native PyTorch object loading - guaranteed to pass weights_only checking
    checkpoint = torch.load(checkpoint_path, weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Restored optimized model state from training epoch {checkpoint['epoch']}")

model.eval()
with torch.no_grad():
    final_recon, final_pred, final_latent = model(val_inputs_X)

y_true_np = val_targets_y.cpu().numpy().flatten()
y_pred_np = final_pred.cpu().numpy().flatten()

X_true_flat = X_val_3d.reshape(-1, feat_dim)
X_recon_flat = final_recon.cpu().numpy().reshape(-1, feat_dim)

recon_r2 = r2_score(X_true_flat, X_recon_flat)
pred_r2 = r2_score(y_true_np, y_pred_np)
pred_rmse = np.sqrt(mean_squared_error(y_true_np, y_pred_np))
pred_mae = mean_absolute_error(y_true_np, y_pred_np)
pred_mape = calculate_mape(y_true_np, y_pred_np)

print("\n================== Final Gannet-Adam Hyper-Optimized Metrics ==================")
print(f"Selected Architecture Layout: {best_num_layers} xLSTM Layers | Alpha Weight: {best_alpha:.4f}")
print(f"Selected Execution Controls : LR: {best_lr:.5f} | Batch Size: {best_batch_size} | Dropout: {best_dropout:.2f}")
print("-" * 79)
print(f"Reconstruction Task Validation R² (X Integrity)     = {recon_r2:.6f}")
print(f"Normalized Prediction Task Validation R² (Carbon Y) = {pred_r2:.6f}")
print(f"Normalized Prediction Task Validation RMSE          = {pred_rmse:.6f}")
print(f"Normalized Prediction Task Validation MAE           = {pred_mae:.6f}")
print(f"Normalized Prediction Task Validation MAPE          = {pred_mape:.2f}%")
print(f"Latent Structural Tensor Dimensionality            = {final_latent.shape}")
print("===============================================================================")

Restored optimized model state from training epoch 14

================== Final Gannet-Adam Hyper-Optimized Metrics ==================
Selected Architecture Layout: 1 xLSTM Layers | Alpha Weight: 2.0000
Selected Execution Controls : LR: 0.00504 | Batch Size: 16 | Dropout: 0.22
-------------------------------------------------------------------------------
Reconstruction Task Validation R² (X Integrity)     = 0.960032
Normalized Prediction Task Validation R² (Carbon Y) = 0.410408
Normalized Prediction Task Validation RMSE          = 0.111260
Normalized Prediction Task Validation MAE           = 0.088352
Normalized Prediction Task Validation MAPE          = 18.45%
Latent Structural Tensor Dimensionality            = torch.Size([600, 16])
